# CRIVO — Exportar a v3 D para NumPy (chat)

Converte os pesos da fase 5 v3, variante D (SHA `a392edec…`), para o formato NumPy do site (float16). Confere se o NumPy dá o mesmo resultado que o PyTorch e divide o arquivo em partes de até 3,5 MB no Drive, para entrar no repositório.

**A v3 D foi reprovada na catraca** (Grok 69,2% < 70,6%; urgência no blind v3 35/42). Ela vai para o chat por decisão do criador, registrada como reprovada.

**Não precisa de GPU.** Ambiente de execução → Executar tudo (uns 2 minutos). Resultado em `MyDrive/CRIVO/fase5v3-numpy/`.

In [ ]:
# Célula 0 — Drive e pesos da v3 D
import os, sys, json, shutil, subprocess, hashlib
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
D_SHA = "a392edec3539fa63df3f38d55af199d18479b8d1625d7199f948f4e2aea8eaf8"
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
ORIGEM = None
for crivo in candidatas:
    for p in sorted((crivo / "transformer-17m-fase5-v3").glob("fase5v3_*/pesos_D.pt")):
        if sha(p) == D_SHA:
            ORIGEM, CRIVO = p, crivo
            break
    if ORIGEM:
        break
if ORIGEM is None:
    raise SystemExit("ERRO: não achei pesos_D.pt (sha a392edec…) em CRIVO/transformer-17m-fase5-v3/. "
                     "Rode na conta onde a v3 treinou.")
LOCAL = Path("/content/fase5v3_d")
shutil.rmtree(LOCAL, ignore_errors=True)
LOCAL.mkdir()
shutil.copyfile(ORIGEM, LOCAL / "pesos.pt")
shutil.copyfile(ORIGEM.parent / "tokenizer.json", LOCAL / "tokenizer.json")
print("pesos da v3 D:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — código do CRIVO e exportação com conferência NumPy × PyTorch
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
subprocess.run([sys.executable, "scripts/exportar_pontuador.py", "--origem", str(LOCAL), "--destino", str(LOCAL)],
               cwd=REPO, check=True)
NPZ = LOCAL / "pesos_numpy.npz"
print("npz:", round(NPZ.stat().st_size / 1e6, 1), "MB | sha256", sha(NPZ), flush=True)


In [ ]:
# Célula 2 — partes de até 3,5 MB no Drive (o conector do Drive não entrega arquivos maiores)
SAIDA = CRIVO / "fase5v3-numpy"
SAIDA.mkdir(parents=True, exist_ok=True)
for antigo in SAIDA.glob("parte_*"):
    antigo.unlink()
dados = NPZ.read_bytes()
TAM = 3584 * 1024
partes = []
for i in range(0, len(dados), TAM):
    nome = "parte_%02d.bin" % (i // TAM)
    (SAIDA / nome).write_bytes(dados[i:i + TAM])
    partes.append({"nome": nome, "sha256": hashlib.sha256(dados[i:i + TAM]).hexdigest()})
manifesto = {"pesos_origem_sha256": D_SHA, "npz_sha256": sha(NPZ), "npz_bytes": len(dados), "partes": partes,
             "tokenizer_sha256": sha(LOCAL / "tokenizer.json")}
(SAIDA / "manifesto_fase5v3.json").write_text(json.dumps(manifesto, indent=1))
print(json.dumps(manifesto, indent=1))
print("\nPRONTO: me avise que eu pego as partes no Drive.", flush=True)
